<a href="https://colab.research.google.com/github/EddyShinee/Copy_Folder_Google_Drive_to_Google_Drive/blob/main/Copy_Folder_Google_Drive_to_Google_Drive_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder Google Drive to Google Drive - 1TouchPro

In [8]:
#@title Input
from ipywidgets import widgets

wide = widgets.Layout(width='100%')
style = {'description_width': '150px'}

dest_text = widgets.Text(
    description="Your drive",
    placeholder='Nhập đường link folder Google Drive của bạn',
    layout=wide,
    style=style,
)
source_count = widgets.BoundedIntText(
    description="Số ô Shared",
    value=20,
    min=1,
    max=20,
    step=1,
    style=style,
)
source_box = widgets.VBox()
source_texts = []

def rebuild_source_inputs(n):
    global source_texts
    old_values = [t.value for t in source_texts]
    source_texts = []
    for i in range(int(n)):
        source_texts.append(widgets.Text(
            description=f"Shared {i + 1}",
            placeholder='Dán 1 link folder shared vào ô này',
            layout=wide,
            style=style,
            value=old_values[i] if i < len(old_values) else '',
        ))
    source_box.children = tuple(source_texts)

source_count.observe(lambda change: rebuild_source_inputs(change['new']), names='value')
rebuild_source_inputs(source_count.value)

from_page_text = widgets.Text(description="Từ trang", value="0", style=style)
to_page_text = widgets.Text(description="Đến trang", value="0", style=style)
max_download_size_text = widgets.Text(description="Tổng dung lượng tối đa(GB)", value="700", style=style)
exclude_str_text = widgets.Text(description="Bỏ file, folder có chứa nội dung", value="", layout=wide, style=style)

display(dest_text)
display(source_count)
display(source_box)
display(from_page_text)
display(to_page_text)
display(max_download_size_text)
display(exclude_str_text)

Text(value='', description='Your drive', layout=Layout(width='100%'), placeholder='Nhập đường link folder Goog…

BoundedIntText(value=20, description='Số ô Shared', max=20, min=1, style=DescriptionStyle(description_width='1…

Text(value='0', description='Từ trang', style=DescriptionStyle(description_width='150px'))

Text(value='0', description='Đến trang', style=DescriptionStyle(description_width='150px'))

Text(value='700', description='Tổng dung lượng tối đa(GB)', style=DescriptionStyle(description_width='150px'))

Text(value='', description='Bỏ file, folder có chứa nội dung', layout=Layout(width='100%'), style=DescriptionS…

In [7]:
#@title Run
import os
import time
import re
import sys
from googleapiclient.discovery import build
from google.colab import auth
from google.colab import drive

class DownloadFromDrive:
    def __init__(self):
        self._total_size = 0
        self._limit_size = 0
        self.excluded_strings = []

    def get_user_credential(self):
        auth.authenticate_user()
        drive_service = build('drive', 'v3')
        return drive_service

    def get_childs_from_folder(self, drive_service, folder_id, from_page, to_page):
        files = []
        page_token = None
        query = f"'{folder_id}' in parents and trashed = false"
        if self.excluded_strings and len(self.excluded_strings) > 0:
            not_contains_query = " and ".join([f"not name contains '{ext}'" for ext in self.excluded_strings])
            query += f" and ({not_contains_query})"

        pages = 0
        while True:
            try:
                pages += 1
                response = drive_service.files().list(q=query,
                                        orderBy='name, createdTime',
                                        fields='files(id, name, mimeType, size), nextPageToken',
                                        pageToken=page_token,
                                        supportsAllDrives=True,
                                        includeItemsFromAllDrives=True).execute()

                if (from_page < pages <= to_page) or to_page == 0:
                    files.extend(response.get('files', []))

                page_token = response.get('nextPageToken', None)
                if page_token is None or  (pages >= to_page > 0):
                    break
            except Exception as e:
                print(f"An error occurred: {str(e)}")
                page_token = None

        print(f"Total files: {len(files)}")
        return files

    def copy_file(self, drive_service, dest_folder_id, source_file):
        if not dest_folder_id:
            print(f"Skip [{source_file['name']}]: destination folder was not created.")
            return

        if source_file['mimeType'] != 'application/vnd.google-apps.folder':
            body_file_inf = {'parents': [dest_folder_id]}

            if not self.check_if_exists(drive_service, dest_folder_id, source_file['name'], is_folder=False):
                try:
                    start_time = time.time()
                    request_copy = drive_service.files().copy(body=body_file_inf, fileId=source_file['id'],
                                                              supportsAllDrives=True).execute()
                    end_time = time.time()

                    fileSize = int(source_file.get('size', 0))
                    size_mb = fileSize / (1024 * 1024)
                    self._total_size += size_mb
                    speed_mb = size_mb / (end_time - start_time)
                    print(f"[{source_file['name']}] copied. Size {size_mb:0.2f} MB. Speed {speed_mb:0.2f} MB/s")


                    if self._total_size >= (self._limit_size * 1024):
                        self.on_total_size_exceeded(f"Total size exceeds {self._limit_size} GB. Ending the program.")
                except Exception as e:
                    print("An error occurred: ", e)
            else:
                print(f"[{source_file['name']}] exists.")
        else:

            source_files = self.get_childs_from_folder(drive_service, source_file['id'], 0, 0)
            if source_files and len(source_files) > 0:
                print(f"Copy at Folder {source_file['name']} Starting")
                sub_folder_id = self.create_folder(drive_service, dest_folder_id, source_file['name'])
                if not sub_folder_id:
                    print(f"Skip Folder {source_file['name']}: could not create destination folder.")
                else:
                    self.copy_multiple_files(drive_service, sub_folder_id, source_files)
                print(f"Copy at Folder {source_file['name']} Ending")


    def create_folder(self, drive_service, dest_folder_id, sub_folder_name):
        if not dest_folder_id:
            print(f"Cannot create folder [{sub_folder_name}]: parent id is empty.")
            return ""

        sub_folder_inf = {'name': sub_folder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_folder_id]}

        exist_folder_id = self.check_if_exists(drive_service, dest_folder_id, sub_folder_name, is_folder=True)
        if exist_folder_id:
            return exist_folder_id
        try:
            folder = drive_service.files().create(body=sub_folder_inf, fields='id', supportsAllDrives=True).execute()
            return folder['id']
        except Exception as e:
            print("An error occurred: ", e)
        return ""


    def check_if_exists(self, drive_service, dest_folder_id, name, is_folder=False):
        try:
            processed_name = name.replace("\\", "\\\\").replace("'", "\\'")
            mime_filter = (
                "mimeType = 'application/vnd.google-apps.folder'"
                if is_folder
                else "mimeType != 'application/vnd.google-apps.folder'"
            )
            # Exact name match in THIS folder only. `name contains` is a
            # substring/full-text search and can hit a same-named file in a
            # subfolder, so the parent file gets skipped as "exists".
            query = (
                f"'{dest_folder_id}' in parents and name = '{processed_name}' "
                f"and {mime_filter} and trashed = false"
            )
            results = drive_service.files().list(
                q=query,
                fields='files(id)',
                pageSize=1,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()

            if 'files' in results and len(results['files']) > 0:
                return results['files'][0]['id']
        except Exception as e:
            print("An error occurred: ", e)

        return ""


    def copy_multiple_files(self, drive_service, dest_folder_id, source_files):
        for source_file in source_files:
            self.copy_file(drive_service, dest_folder_id, source_file)

    def extract_folder_id_from_url(self, url):
        pattern = r'[-\w]{25,}'
        match = re.search(pattern, url)
        if match:
            return match.group(0)
        else:
            return None

    def parse_source_links(self, raw_links):
        links = []
        seen = set()
        for i, part in enumerate(raw_links or [], start=1):
            part = (part or '').strip()
            if not part:
                continue
            folder_id = self.extract_folder_id_from_url(part)
            if not folder_id:
                print(f"Skip invalid link (Shared {i}): {part}")
                continue
            if folder_id in seen:
                print(f"Skip duplicate link (Shared {i})")
                continue
            seen.add(folder_id)
            links.append(part)
        return links

    def on_total_size_exceeded(self, message):
        print(message)
        sys.exit()

    def copy_one_source(self, service, dest_folder_id, sourceDriveLink, from_page, to_page, index=1, total=1):
        source_folder_id = self.extract_folder_id_from_url(sourceDriveLink)
        if not source_folder_id:
            print(f"[{index}/{total}] Skip invalid source link: {sourceDriveLink}")
            return False

        try:
            source_folder = service.files().get(fileId=source_folder_id, supportsAllDrives=True).execute()
        except Exception as e:
            print(f"[{index}/{total}] Cannot read source folder: {e}")
            return False

        folder_name = source_folder.get('name', source_folder_id)
        print(f"\n===== [{index}/{total}] START {folder_name} =====")
        new_dest_folder_id = self.create_folder(service, dest_folder_id, folder_name)
        if not new_dest_folder_id:
            print(f"[{index}/{total}] Skip {folder_name}: could not create destination folder.")
            return False

        source_files = self.get_childs_from_folder(service, source_folder_id, from_page, to_page)
        self.copy_multiple_files(service, new_dest_folder_id, source_files)
        print(f"===== [{index}/{total}] DONE {folder_name} =====")
        return True

    def copy_drive_to_drive(self, destDriveLink, sourceDriveLinks, from_page, to_page):
        service = self.get_user_credential()

        dest_folder_id = self.extract_folder_id_from_url(destDriveLink)
        if not dest_folder_id:
            print("Invalid destination folder link.")
            return

        source_links = self.parse_source_links(sourceDriveLinks)
        if not source_links:
            print("Chưa nhập link Shared folder hợp lệ.")
            return

        start_time = time.time()
        ok = 0
        for i, source_link in enumerate(source_links, start=1):
            try:
                if self.copy_one_source(service, dest_folder_id, source_link, from_page, to_page, i, len(source_links)):
                    ok += 1
            except SystemExit:
                raise
            except Exception as e:
                print(f"[{i}/{len(source_links)}] Failed: {e}")

        end_time = time.time()
        size_gb = self._total_size / 1024
        elapsed = max(end_time - start_time, 1e-6)
        speed_mb = self._total_size / elapsed
        print(
            f"\nDone {ok}/{len(source_links)} folder(s). "
            f"Total Size {size_gb:0.2f} GB. Total Time {int(elapsed)} s. SpeedMB {speed_mb:0.2f} MB/s"
        )


# Main
destDriveLink = dest_text.value
sourceDriveLinks = [t.value for t in source_texts]
fromPage = int(from_page_text.value)
toPage = int(to_page_text.value)

downloader = DownloadFromDrive()
downloader._limit_size = float(max_download_size_text.value);
downloader.excluded_strings = [ext.strip() for ext in exclude_str_text.value.split(",") if ext.strip()]
downloader.copy_drive_to_drive(destDriveLink, sourceDriveLinks, fromPage, toPage)

Total files: 3
[[VIP] Danh Sách Tổng Hợp Khóa Học Online - Tài Liệu - Ebook 2024.xlsx] copied. Size 1.44 MB. Speed 0.85 MB/s
Total files: 14
Copy at Folder CFA LEVEL 1 (HEDGE ACADEMY) Starting
Total files: 6
Copy at Folder 00. Tài liệu học CFA Starting
Total files: 18
Copy at Folder Bộ Giáo trình + Từ điển đọc thêm Starting
[.DS_Store] copied. Size 0.01 MB. Speed 0.01 MB/s
[L1-2020-Alt Inv _ Port Mgt.pdf] copied. Size 9.45 MB. Speed 7.14 MB/s
[L1-2020-Corporate Fin _ Equity.pdf] copied. Size 49.01 MB. Speed 49.73 MB/s
[L1-2020-Economics.pdf] copied. Size 7.15 MB. Speed 6.84 MB/s
[L1-2020-Ethics _ QM.pdf] copied. Size 10.24 MB. Speed 9.03 MB/s
[L1-2020-Fixed Inc _ Derivatives.pdf] copied. Size 8.17 MB. Speed 6.72 MB/s
[L1-2020-FRA.pdf] copied. Size 84.40 MB. Speed 57.15 MB/s
[Mind Maps CFA Level 1.pdf] copied. Size 1.54 MB. Speed 0.92 MB/s
[Schweser_s Quicksheet.pdf] copied. Size 3.97 MB. Speed 3.64 MB/s
[Study Plan CFA Level 1.pdf] copied. Size 0.07 MB. Speed 0.

An error occurred:  <HttpError 403 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "The specified parent is not a folder.". Details: "[{'message': 'The specified parent is not a folder.', 'domain': 'global', 'reason': 'parentNotAFolder'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?q=%27%27+in+parents+and+name+contains+%27B%E1%BA%A3n+sao+c%E1%BB%A7a+Participation+-+CFA+Level+2+-+Class+A002+AUG+2021+%5BHedge+Academy%5D.xlsx%27+and+trashed%3Dfalse&fields=files%28id%29&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files/1x8T-EwuOLYFvnJYOn_9rJQN7sjtQulsz/copy?supportsAllDrives=true&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 're

An error occurred:  <HttpError 403 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "The specified parent is not a folder.". Details: "[{'message': 'The specified parent is not a folder.', 'domain': 'global', 'reason': 'parentNotAFolder'}]">
Total files: 6
Copy at Folder CFA LEVEL 2 - HEDGE ACADEMY Starting
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?q=%27%27+in+parents+and+name+contains+%27CFA+LEVEL+2+-+HEDGE+ACADEMY%27+and+trashed%3Dfalse&fields=files%28id%29&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'pa

An error occurred:  <HttpError 403 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "The specified parent is not a folder.". Details: "[{'message': 'The specified parent is not a folder.', 'domain': 'global', 'reason': 'parentNotAFolder'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?q=%27%27+in+parents+and+name+contains+%27Curriculum+2022+-+CFA+Level+II+Box+Set+%28vol.+1-6%29+-+Wiley+%282021%29.pdf%27+and+trashed%3Dfalse&fields=files%28id%29&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files/1zI1vPCmj4mGwR1rXcfhBJyXBsuwRsX8C/copy?supportsAllDrives=true&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'locati

An error occurred:  <HttpError 403 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "The specified parent is not a folder.". Details: "[{'message': 'The specified parent is not a folder.', 'domain': 'global', 'reason': 'parentNotAFolder'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?q=%27%27+in+parents+and+name+contains+%27CFA+2023+Level+II+-+Schweser_s+Quicksheet%281%29.pdf%27+and+trashed%3Dfalse&fields=files%28id%29&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files/1OF7KQqSDBKuZOEqBkfYNbmWnhpgOsB4f/copy?supportsAllDrives=true&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'location

An error occurred:  <HttpError 403 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "The specified parent is not a folder.". Details: "[{'message': 'The specified parent is not a folder.', 'domain': 'global', 'reason': 'parentNotAFolder'}]">
Total files: 0
Total files: 10
Copy at Folder 3. Equity Starting
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?q=%27%27+in+parents+and+name+contains+%273.+Equity%27+and+trashed%3Dfalse&fields=files%28id%29&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An erro

An error occurred:  <HttpError 403 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "The specified parent is not a folder.". Details: "[{'message': 'The specified parent is not a folder.', 'domain': 'global', 'reason': 'parentNotAFolder'}]">
Total files: 5
Copy at Folder 10. AI Starting
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?q=%27%27+in+parents+and+name+contains+%2710.+AI%27+and+trashed%3Dfalse&fields=files%28id%29&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <HttpErro

An error occurred:  <HttpError 403 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "The specified parent is not a folder.". Details: "[{'message': 'The specified parent is not a folder.', 'domain': 'global', 'reason': 'parentNotAFolder'}]">
Total files: 3
Copy at Folder 1. Quant Starting
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?q=%27%27+in+parents+and+name+contains+%271.+Quant%27+and+trashed%3Dfalse&fields=files%28id%29&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?fields=id&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">
An error occurred:  <Http